# XGBoost Symptom Classifier — Training

This notebook trains an XGBoost classifier that predicts mango disease from a binary symptom feature vector.
It is **standalone** — independent of the Django project — and is intended to run on Google Colab.

**Input**: CSV file generated by notebook `01_generate_training_data.ipynb`  
**Output**: `symptom_classifier_{disease_type}.json` + `symptom_classifier_{disease_type}_meta.json`

**Workflow**:
1. Load and validate CSV data
2. Encode binary symptom feature vectors
3. Balance classes with SMOTE if needed
4. Train XGBoost with early stopping
5. Evaluate on held-out test set
6. Save model + metadata for deployment in the Django backend

In [ ]:
!pip install xgboost scikit-learn imbalanced-learn pandas numpy matplotlib seaborn

In [ ]:
import os
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
print('All imports successful.')

## Configuration

In [ ]:
# ── Change DISEASE_TYPE to 'fruit' to train the fruit model ──────────────────
DISEASE_TYPE = 'leaf'  # change to 'fruit' for fruit model

DATA_CSV = 'leaf_training_data.csv'  # update path as needed
MIN_SAMPLES_PER_CLASS = 10
OUTPUT_DIR = '.'
RANDOM_SEED = 42

# ── Symptom vocabularies (order = feature-vector index) ─────────────────────
LEAF_SYMPTOMS = [
    'dark_spots_brown',
    'dark_spots_with_yellow_halo',
    'concentric_rings_on_lesion',
    'black_specks_in_lesion',
    'pink_spore_masses',
    'lesions_enlarge_rapidly',
    'spots_with_irregular_border',
    'twig_dieback_from_tip',
    'canker_lesions_on_twig',
    'bark_cracking',
    'wilting_shoot_tips',
    'sparse_foliage',
    'white_powder_coating',
    'leaf_distortion',
    'premature_leaf_drop',
    'black_sooty_coating',
    'sooty_deposit_wiped_off',
    'yellow_discoloration',
    'brown_leaf_margins',
    'brown_leaf_tips',
    'water_soaked_lesions',
    'leaf_curling',
]

FRUIT_SYMPTOMS = [
    'black_sunken_lesions',
    'brown_patches_spreading',
    'surface_cracks_radiating',
    'pink_spore_masses_on_lesion',
    'lesion_ring_pattern',
    'soft_rot_spreading',
    'stem_end_rot',
    'shriveling_of_fruit',
    'premature_fruit_drop',
    'white_powder_on_fruit',
    'black_sooty_deposit_on_fruit',
    'fruit_discoloration',
]

LEAF_DISEASES  = ['Anthracnose', 'Die Back', 'Healthy', 'Powdery Mildew', 'Sooty Mold']
FRUIT_DISEASES = ['Anthracnose', 'Healthy']

# ── Resolve active vocabulary & disease list based on DISEASE_TYPE ───────────
if DISEASE_TYPE == 'leaf':
    VOCABULARY = LEAF_SYMPTOMS
    DISEASES   = LEAF_DISEASES
    if DATA_CSV == 'leaf_training_data.csv' and DISEASE_TYPE == 'fruit':
        DATA_CSV = 'fruit_training_data.csv'
elif DISEASE_TYPE == 'fruit':
    VOCABULARY = FRUIT_SYMPTOMS
    DISEASES   = FRUIT_DISEASES
    DATA_CSV   = 'fruit_training_data.csv'
else:
    raise ValueError(f"Unknown DISEASE_TYPE: {DISEASE_TYPE!r}. Must be 'leaf' or 'fruit'.")

print(f'DISEASE_TYPE         : {DISEASE_TYPE}')
print(f'DATA_CSV             : {DATA_CSV}')
print(f'Vocabulary size      : {len(VOCABULARY)}')
print(f'Disease classes      : {DISEASES}')
print(f'MIN_SAMPLES_PER_CLASS: {MIN_SAMPLES_PER_CLASS}')
print(f'OUTPUT_DIR           : {OUTPUT_DIR}')

## Step 1: Load Data

In [ ]:
# ── Load CSV ─────────────────────────────────────────────────────────────────
df_raw = pd.read_csv(DATA_CSV)
print(f'Raw CSV shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')

# ── Filter required columns ───────────────────────────────────────────────────
required_cols = {'selected_symptoms', 'disease_classification', 'disease_type'}
missing = required_cols - set(df_raw.columns)
if missing:
    raise ValueError(f'CSV is missing required columns: {missing}')

# ── Filter by disease_type ────────────────────────────────────────────────────
df = df_raw[df_raw['disease_type'] == DISEASE_TYPE].copy()
print(f'Rows matching disease_type={DISEASE_TYPE!r}: {len(df)}')

# ── Drop rows with null/empty disease_classification ─────────────────────────
df = df[df['disease_classification'].notna()]
df = df[df['disease_classification'].astype(str).str.strip() != '']

# ── Drop rows with null/empty selected_symptoms ───────────────────────────────
df = df[df['selected_symptoms'].notna()]
df = df[df['selected_symptoms'].astype(str).str.strip().isin(['', '[]', 'null']) == False]

# ── Parse selected_symptoms from JSON string to Python list ──────────────────
def parse_symptoms(val):
    if isinstance(val, list):
        return val
    try:
        parsed = json.loads(val)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

df['selected_symptoms'] = df['selected_symptoms'].apply(parse_symptoms)
df = df[df['selected_symptoms'].apply(lambda x: len(x) > 0)]

print(f'Filtered shape (valid rows): {df.shape}')
print()
df.head()

## Step 2: Validate Class Distribution

In [ ]:
# ── Count samples per class ───────────────────────────────────────────────────
class_counts = df['disease_classification'].value_counts()
print('Class distribution:')
print('-' * 40)
for cls, cnt in class_counts.items():
    status = 'PASS' if cnt >= MIN_SAMPLES_PER_CLASS else 'FAIL'
    print(f'  [{status}]  {cls:<25} {cnt:>5} samples')
print('-' * 40)

# ── Determine eligible classes ────────────────────────────────────────────────
eligible_classes = set(
    cls for cls, cnt in class_counts.items() if cnt >= MIN_SAMPLES_PER_CLASS
)
print(f'\nEligible classes (>= {MIN_SAMPLES_PER_CLASS} samples): {sorted(eligible_classes)}')

# ── Guard: need at least 2 eligible classes ───────────────────────────────────
if len(eligible_classes) < 2:
    raise ValueError(
        f'FAIL: Only {len(eligible_classes)} class(es) meet the minimum sample '
        f'requirement of {MIN_SAMPLES_PER_CLASS}. Need at least 2. '
        f'Collect more training data and regenerate the CSV.'
    )

print(f'\nPASS: {len(eligible_classes)} eligible classes found. Proceeding to training.')

# ── Filter dataframe to eligible classes only ─────────────────────────────────
df = df[df['disease_classification'].isin(eligible_classes)].reset_index(drop=True)
print(f'Dataset after filtering to eligible classes: {len(df)} rows')

## Step 3: Feature Encoding

In [ ]:
class SymptomFeatureEncoder:
    """Encodes a list of symptom strings into a binary float32 feature vector."""

    def __init__(self, vocabulary: list):
        self.vocabulary = vocabulary
        self.symptom_to_index = {sym: i for i, sym in enumerate(vocabulary)}
        self.n_features = len(vocabulary)

    def encode(self, symptom_list: list) -> np.ndarray:
        """Return a binary float32 vector of shape (n_features,)."""
        vec = np.zeros(self.n_features, dtype=np.float32)
        for sym in symptom_list:
            idx = self.symptom_to_index.get(sym)
            if idx is not None:
                vec[idx] = 1.0
        return vec


# ── Deterministic label→int mapping (sorted for reproducibility) ──────────────
sorted_classes = sorted(eligible_classes)
label_to_int   = {cls: i for i, cls in enumerate(sorted_classes)}
int_to_label   = {i: cls for cls, i in label_to_int.items()}

print('Label → integer mapping:')
for cls, idx in label_to_int.items():
    print(f'  {idx}  →  {cls}')

# ── Build feature matrix X and label array y ──────────────────────────────────
encoder = SymptomFeatureEncoder(vocabulary=VOCABULARY)

X = np.vstack(df['selected_symptoms'].apply(encoder.encode).values)
y = np.array([label_to_int[cls] for cls in df['disease_classification']], dtype=np.int32)

print(f'\nX shape: {X.shape}  (samples × features)')
print(f'y shape: {y.shape}')
print(f'Feature dtype: {X.dtype}')

## Step 4: Class Balance Check & SMOTE

In [ ]:
# ── Compute minority ratio ────────────────────────────────────────────────────
class_bin_counts = np.bincount(y)
minority_count   = class_bin_counts.min()
majority_count   = class_bin_counts.max()
total_samples    = len(y)
minority_ratio   = minority_count / majority_count

print('Class distribution (before SMOTE):')
for i, cnt in enumerate(class_bin_counts):
    print(f'  Class {i} ({int_to_label[i]}): {cnt} samples')
print(f'\nMinority / Majority ratio: {minority_ratio:.3f}')
print(f'Total samples            : {total_samples}')

# ── Bar chart of class distribution ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    [int_to_label[i] for i in range(len(class_bin_counts))],
    class_bin_counts,
    color='steelblue',
    edgecolor='black',
)
ax.set_title('Class Distribution Before SMOTE')
ax.set_xlabel('Disease Class')
ax.set_ylabel('Sample Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

# ── Apply SMOTE if imbalanced ─────────────────────────────────────────────────
APPLY_SMOTE = (minority_ratio < 0.3) and (total_samples >= 20)

if APPLY_SMOTE:
    print(f'\nApplying SMOTE (minority_ratio={minority_ratio:.3f} < 0.3) ...')
    smote = SMOTE(random_state=RANDOM_SEED)
    X, y = smote.fit_resample(X, y)
    after_counts = np.bincount(y)
    print('Class distribution after SMOTE:')
    for i, cnt in enumerate(after_counts):
        print(f'  Class {i} ({int_to_label[i]}): {cnt} samples')
    print(f'Total samples after SMOTE: {len(y)}')
else:
    reason = 'minority_ratio >= 0.3' if minority_ratio >= 0.3 else f'total_samples ({total_samples}) < 20'
    print(f'\nSMOTE skipped ({reason}).')

## Step 5: Train/Val/Test Split

In [ ]:
# ── First split: hold out 15% as test set ────────────────────────────────────
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_SEED,
)

# ── Second split: 15/85 val from remaining (≈15% of original total) ──────────
val_fraction_of_trainval = 0.15 / 0.85  # ~0.1765
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_fraction_of_trainval,
    stratify=y_trainval,
    random_state=RANDOM_SEED,
)

print('Split sizes:')
print(f'  Train : {len(X_train):>6} samples  ({100*len(X_train)/len(X):.1f}%)')
print(f'  Val   : {len(X_val):>6} samples  ({100*len(X_val)/len(X):.1f}%)')
print(f'  Test  : {len(X_test):>6} samples  ({100*len(X_test)/len(X):.1f}%)')
print(f'  Total : {len(X):>6} samples')

## Step 6: Train XGBoost

In [ ]:
n_classes = len(sorted_classes)
objective = 'binary:logistic' if n_classes == 2 else 'multi:softprob'

model = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    device='cpu',
    random_state=RANDOM_SEED,
    eval_metric='mlogloss',
    early_stopping_rounds=20,
    objective=objective,
    num_class=n_classes if n_classes > 2 else None,
    use_label_encoder=False,
)

print(f'Training XGBClassifier  ({n_classes} classes, objective={objective}) ...')

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print(f'Training complete. Best iteration: {model.best_iteration}')

## Step 7: Evaluate

In [ ]:
# ── Predict on test set ───────────────────────────────────────────────────────
y_pred = model.predict(X_test)

# ── Overall accuracy ──────────────────────────────────────────────────────────
test_accuracy = accuracy_score(y_test, y_pred)

# ── Per-class accuracy (boolean mask approach) ────────────────────────────────
per_class_accuracy = {}
for class_idx, class_name in int_to_label.items():
    mask = y_test == class_idx
    if mask.sum() > 0:
        cls_acc = accuracy_score(y_test[mask], y_pred[mask])
        per_class_accuracy[class_name] = round(float(cls_acc), 4)
    else:
        per_class_accuracy[class_name] = None

# ── Classification report ─────────────────────────────────────────────────────
target_names = [int_to_label[i] for i in range(n_classes)]
report = classification_report(y_test, y_pred, target_names=target_names)
print(report)

print('-' * 40)
print(f'Test accuracy: {test_accuracy * 100:.2f}%')
print('Per-class accuracy:')
for cls, acc in per_class_accuracy.items():
    acc_str = f'{acc * 100:.2f}%' if acc is not None else 'N/A (no test samples)'
    print(f'  {cls:<25} {acc_str}')

In [ ]:
# ── Confusion matrix heatmap ──────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(max(6, n_classes * 1.5), max(5, n_classes * 1.2)))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=target_names,
    yticklabels=target_names,
    ax=ax,
)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title(f'Confusion Matrix — {DISEASE_TYPE.capitalize()} Disease Classifier')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

cm_path = os.path.join(OUTPUT_DIR, f'confusion_matrix_{DISEASE_TYPE}.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Confusion matrix saved to: {cm_path}')

## Step 8: Feature Importances

In [ ]:
# ── Extract and rank feature importances ─────────────────────────────────────
importances = model.feature_importances_  # shape: (n_features,)
feature_importance_pairs = list(zip(VOCABULARY, importances.tolist()))
feature_importance_pairs.sort(key=lambda x: x[1], reverse=True)

TOP_N = 15
top_features_pairs = feature_importance_pairs[:TOP_N]
top_features = [{'symptom': sym, 'importance': round(imp, 6)} for sym, imp in top_features_pairs]

# ── Horizontal bar chart ──────────────────────────────────────────────────────
top_syms  = [f['symptom']    for f in top_features]
top_imps  = [f['importance'] for f in top_features]

fig, ax = plt.subplots(figsize=(9, max(5, TOP_N * 0.45)))
bars = ax.barh(
    range(len(top_syms)),
    top_imps[::-1],   # reverse so highest is at top
    color='darkorange',
    edgecolor='black',
)
ax.set_yticks(range(len(top_syms)))
ax.set_yticklabels(top_syms[::-1])
ax.set_xlabel('Feature Importance Score')
ax.set_title(f'Top {TOP_N} Feature Importances — {DISEASE_TYPE.capitalize()} Classifier')
plt.tight_layout()

fi_path = os.path.join(OUTPUT_DIR, f'feature_importances_{DISEASE_TYPE}.png')
plt.savefig(fi_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Feature importance chart saved to: {fi_path}')

print(f'\nTop {TOP_N} features:')
for rank, feat in enumerate(top_features, 1):
    print(f'  {rank:>2}. {feat["symptom"]:<35} {feat["importance"]:.6f}')

## Step 9: Save Model & Metadata

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Save XGBoost model (native JSON format) ───────────────────────────────────
model_filename = f'symptom_classifier_{DISEASE_TYPE}.json'
model_path     = os.path.join(OUTPUT_DIR, model_filename)
model.save_model(model_path)

# ── Build metadata dict ───────────────────────────────────────────────────────
meta = {
    'disease_type'       : DISEASE_TYPE,
    'classes'            : sorted_classes,
    'label_to_int'       : label_to_int,
    'vocabulary'         : VOCABULARY,
    'n_samples'          : int(len(y)),
    'test_accuracy'      : round(float(test_accuracy), 6),
    'per_class_accuracy' : per_class_accuracy,
    'top_features'       : top_features,
    'training_date'      : datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
    'hyperparameters': {
        'n_estimators'      : 150,
        'max_depth'         : 5,
        'learning_rate'     : 0.1,
        'subsample'         : 0.8,
        'colsample_bytree'  : 0.8,
        'tree_method'       : 'hist',
        'device'            : 'cpu',
        'eval_metric'       : 'mlogloss',
        'early_stopping_rounds': 20,
        'random_state'      : RANDOM_SEED,
        'best_iteration'    : int(model.best_iteration),
    },
    'smote_applied'      : APPLY_SMOTE,
}

# ── Save metadata ─────────────────────────────────────────────────────────────
meta_filename = f'symptom_classifier_{DISEASE_TYPE}_meta.json'
meta_path     = os.path.join(OUTPUT_DIR, meta_filename)
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

print(f'Model saved : {model_path}')
print(f'Meta saved  : {meta_path}')
print(f'Done! Test accuracy: {test_accuracy * 100:.2f}%')

## (Optional) Mount Google Drive to Persist Model

Run the cell below to mount your Google Drive and copy the trained model files there so they survive the Colab session.

After mounting, the files will be copied to:
```
/content/drive/MyDrive/MangoSense/models/
```

You can then download them from Drive and place them in the Django project's `models/` directory.

In [ ]:
import shutil

DRIVE_SAVE_DIR = '/content/drive/MyDrive/MangoSense/models'

try:
    from google.colab import drive
    drive.mount('/content/drive')

    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

    # Copy model file
    dest_model = os.path.join(DRIVE_SAVE_DIR, model_filename)
    shutil.copy2(model_path, dest_model)
    print(f'Copied model to Drive: {dest_model}')

    # Copy metadata file
    dest_meta = os.path.join(DRIVE_SAVE_DIR, meta_filename)
    shutil.copy2(meta_path, dest_meta)
    print(f'Copied metadata to Drive: {dest_meta}')

    # Copy charts if they exist
    for chart_name in [f'confusion_matrix_{DISEASE_TYPE}.png', f'feature_importances_{DISEASE_TYPE}.png']:
        src = os.path.join(OUTPUT_DIR, chart_name)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_SAVE_DIR, chart_name))
            print(f'Copied chart to Drive: {chart_name}')

    print(f'\nAll files saved to Google Drive at: {DRIVE_SAVE_DIR}')

except ImportError:
    print('Not running in Google Colab — skipping Drive mount.')
except Exception as e:
    print(f'Drive mount/copy failed: {e}')
    print('You can manually download the files from the Colab file browser on the left.')

## Summary & Next Steps

### What was produced

| File | Description |
|---|---|
| `symptom_classifier_{disease_type}.json` | Trained XGBoost model (native JSON format) |
| `symptom_classifier_{disease_type}_meta.json` | Metadata: classes, vocabulary, accuracy, top features, hyperparams |
| `confusion_matrix_{disease_type}.png` | Confusion matrix heatmap on held-out test set |
| `feature_importances_{disease_type}.png` | Top-15 feature importance bar chart |

### Next steps

1. **Test the model** — Open notebook `03_test_symptom_classifier.ipynb` to run inference on individual symptom lists and verify predictions.

2. **Deploy to Django** — Copy `symptom_classifier_{disease_type}.json` and `symptom_classifier_{disease_type}_meta.json` into the Django project's `models/` directory:
   ```
   manggo-backend/
   └── models/
       ├── symptom_classifier_leaf.json
       ├── symptom_classifier_leaf_meta.json
       ├── symptom_classifier_fruit.json
       └── symptom_classifier_fruit_meta.json
   ```

3. **Train the fruit model** — Set `DISEASE_TYPE = 'fruit'` and `DATA_CSV = 'fruit_training_data.csv'` in the Configuration cell, then re-run the notebook.

4. **Retrain with more data** — As more verified predictions accumulate in the admin dashboard, re-export the CSV and retrain to improve accuracy.